# Phase 10: Exploratory Data Analysis (EDA)
## Marketplace Profitability & Delivery Intelligence Engine

This notebook explores the operational, financial, and customer experience dynamics of the Olist e-commerce marketplace.
Data is read directly from the processed analytical feature store (`data/processed/order_features.parquet`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

# Load processed feature store and enriched reviews
orders = pd.read_parquet('../data/processed/order_features.parquet')
reviews = pd.read_parquet('../data/processed/order_reviews_enriched.parquet')
merged = pd.merge(orders, reviews[['order_id', 'review_score', 'csat_category', 'is_delivery_issue']], on='order_id', how='left')
delivered = merged[merged['is_delivered']].copy()
print(f'Loaded {len(orders):,} total orders ({len(delivered):,} delivered orders) with {len(orders.columns)} features.')

### 1. Delivery Delay vs Customer Satisfaction (CSAT)
**Business Question**: How does delivery timeliness affect customer review scores, and what is the tolerance threshold before reviews collapse to 1 star?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
delay_clipped = delivered['delivery_delay_days'].clip(-25, 25)
palette = ['#d9534f', '#f0ad4e', '#f7ecb5', '#5bc0de', '#5cb85c']
sns.boxplot(x=delivered['review_score'], y=delay_clipped, palette=palette, hue=delivered['review_score'], legend=False, ax=ax, boxprops=dict(alpha=0.85))
ax.axhline(0, color='black', linestyle='--', linewidth=1.2, label='Promised SLA (0 Days)')
ax.set_title('Customer Review Score vs Delivery Delay (Days Relative to Estimate)', pad=12)
ax.set_xlabel('Customer Review Score (Stars)')
ax.set_ylabel('Delivery Delay (Days: Negative = Early, Positive = Late)')
ax.legend(loc='upper right')
plt.show()

### 2. Freight-to-Price Friction by Product Category
**Business Question**: Which merchandise categories experience the highest shipping cost friction relative to item value?

In [ ]:
cat_freight = delivered.groupby('primary_product_category').agg(
    order_count=('order_id', 'count'),
    median_freight_pct=('freight_ratio', lambda s: s.median() * 100.0)
).query('order_count >= 500').sort_values('median_freight_pct').tail(12)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(cat_freight.index, cat_freight['median_freight_pct'], color=plt.cm.YlOrRd(np.linspace(0.4, 0.9, len(cat_freight))))
ax.set_title('Median Freight Cost as % of Total Order Value (Top Friction Categories)', pad=12)
ax.set_xlabel('Median Freight Ratio (%)')
for b in bars:
    ax.text(b.get_width() + 0.3, b.get_y() + b.get_height()/2, f'{b.get_width():.1f}%', va='center')
plt.show()

### 3. Seller Volume vs Delivery Delay Rate (Log Scale)
**Business Question**: Does merchant fulfillment scale improve delivery timeliness or create logistical bottlenecks?

In [ ]:
seller_stats = delivered.groupby('primary_seller_id').agg(
    orders=('order_id', 'count'),
    late_rate=('is_late_delivery', lambda s: s.mean() * 100.0),
    gmv=('order_total_item_value', 'sum')
).query('orders >= 10')

fig, ax = plt.subplots(figsize=(10, 5.5))
sc = ax.scatter(seller_stats['orders'], seller_stats['late_rate'], c=np.log10(seller_stats['gmv'] + 1), cmap='viridis', alpha=0.65, s=40)
plt.colorbar(sc, ax=ax, label='Log10 Total GMV (BRL)')
ax.set_xscale('log')
ax.axhline(8.11, color='red', linestyle='--', label='Marketplace Avg Late Rate (8.11%)')
ax.set_title('Seller Order Volume vs Late Delivery Rate (Log Scale)', pad=12)
ax.set_xlabel('Fulfilled Order Volume (Log Scale)')
ax.set_ylabel('Late Delivery Rate (%)')
ax.legend()
plt.show()

### 4. Top Categories: Gross GMV vs Net Contribution Margin
**Business Question**: Which categories drive the highest top-line GMV, and which generate the highest net cash contribution after fees and delivery remedy vouchers?

In [ ]:
cat_fin = delivered.groupby('primary_product_category').agg(
    gmv=('order_total_item_value', 'sum'),
    late_orders=('is_late_delivery', 'sum'),
    orders=('order_id', 'count')
).query('orders >= 1000').copy()
cat_fin['net_margin'] = (cat_fin['gmv'] * 0.12) - (cat_fin['gmv'] * 0.018) - (cat_fin['late_orders'] * 15.0)
cat_top = cat_fin.sort_values('gmv', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 5.5))
x = np.arange(len(cat_top))
w = 0.35
ax.bar(x - w/2, cat_top['gmv'] / 1000.0, w, label='Gross GMV (k BRL)', color='#2b5c8f')
ax.bar(x + w/2, cat_top['net_margin'] / 1000.0, w, label='Net Contribution (k BRL)', color='#4ca64c')
ax.set_xticks(x)
ax.set_xticklabels(cat_top.index, rotation=35, ha='right')
ax.set_title('Top 10 Product Categories: Gross GMV vs Net Platform Margin', pad=12)
ax.set_ylabel('Thousand BRL (k R$)')
ax.legend()
plt.show()

### 5. Geographic Regional Logistics Performance
**Business Question**: How does delivery speed and SLA failure rate vary across Brazil's 5 macro-regions?

In [ ]:
state_map = {
    'SP': 'Southeast', 'RJ': 'Southeast', 'MG': 'Southeast', 'ES': 'Southeast',
    'PR': 'South', 'SC': 'South', 'RS': 'South',
    'BA': 'Northeast', 'PE': 'Northeast', 'CE': 'Northeast', 'MA': 'Northeast', 'PB': 'Northeast',
    'RN': 'Northeast', 'AL': 'Northeast', 'SE': 'Northeast', 'PI': 'Northeast',
    'DF': 'Central-West', 'GO': 'Central-West', 'MT': 'Central-West', 'MS': 'Central-West',
    'AM': 'North', 'PA': 'North', 'RO': 'North', 'AC': 'North', 'TO': 'North', 'AP': 'North', 'RR': 'North',
}
delivered['macro_region'] = delivered['customer_state'].map(state_map).fillna('Unknown')
reg = delivered.groupby('macro_region').agg(
    avg_days=('actual_delivery_duration_days', 'mean'),
    late_pct=('is_late_delivery', lambda s: s.mean() * 100.0)
).sort_values('avg_days')

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.bar(reg.index, reg['avg_days'], color=['#4a90e2', '#50e3c2', '#b8e986', '#f5a623', '#d0021b'], alpha=0.85, width=0.45)
ax2.plot(reg.index, reg['late_pct'], color='#8b0000', marker='o', linewidth=2.5, label='Late Delivery Rate (%)')
ax1.set_title('Delivery Transit Days vs Late Delivery Rate by Brazil Macro-Region', pad=12)
ax1.set_ylabel('Average Delivery Transit (Days)')
ax2.set_ylabel('Late Delivery Rate (%)')
ax2.grid(False)
ax2.legend(loc='upper left')
plt.show()

### 6. Root-Cause Attribution: Seller Processing Delay vs Carrier Transit Duration
**Business Question**: When an order is delivered late, is the bottleneck caused by the seller's internal dispatch delay or carrier transit?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
seller_days = delivered['seller_dispatch_duration_hours'] / 24.0
sns.kdeplot(x=seller_days, hue=delivered['is_late_delivery'], common_norm=False, palette={0.0: '#2b5c8f', 1.0: '#d9534f'}, clip=(0, 15), fill=True, alpha=0.3, ax=ax)
ax.set_title('Kernel Density of Seller Dispatch Processing Duration (Days)', pad=12)
ax.set_xlabel('Seller Dispatch Duration (Days from Purchase to Carrier Handoff)')
ax.set_ylabel('Density')
ax.legend(labels=['Late Delivery', 'On-Time Delivery'], loc='upper right')
plt.show()

### 7. Merchant Revenue Concentration: Lorenz Curve & Gini Coefficient
**Business Question**: How severe is seller revenue concentration in the marketplace?

In [ ]:
seller_rev = delivered.groupby('primary_seller_id')['order_total_item_value'].sum().sort_values().values
cum_rev = np.cumsum(seller_rev) / np.sum(seller_rev)
cum_sellers = np.linspace(0, 1, len(seller_rev))
b_area = np.trapezoid(cum_rev, cum_sellers)
gini = (0.5 - b_area) / 0.5

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.plot(cum_sellers * 100.0, cum_rev * 100.0, color='#2b5c8f', linewidth=2.5, label=f'Marketplace Lorenz Curve (Gini = {gini:.2f})')
ax.plot([0, 100], [0, 100], color='gray', linestyle='--', label='Perfect Equality Line')
ax.scatter([80.2], [20.0], color='red', s=50, zorder=5)
ax.annotate('Pareto Point:\n80.2% of Sellers = 20% GMV', (80.2, 20.0), xytext=(50, 30), arrowprops=dict(arrowstyle='->', color='red'))
ax.set_title('Merchant Revenue Concentration: Lorenz Curve & Gini Coefficient', pad=12)
ax.set_xlabel('Cumulative % of Sellers (Smallest to Largest)')
ax.set_ylabel('Cumulative % of Marketplace GMV')
ax.legend(loc='upper left')
plt.show()

### 8. Customer Lifetime Spend: One-Time vs Repeat Buyers
**Business Question**: What is the commercial value difference between one-time and repeat buyers?

In [ ]:
cust_spend = orders.groupby('customer_unique_id').agg(
    order_count=('order_id', 'count'),
    total_spend=('order_total_value', 'sum')
)
cust_spend['segment'] = np.where(cust_spend['order_count'] > 1, 'Repeat Buyer (3.1%)', 'One-Time Buyer (96.9%)')

fig, ax = plt.subplots(figsize=(9, 5.5))
sns.boxplot(data=cust_spend, x='segment', y='total_spend', palette=['#4a90e2', '#f5a623'], hue='segment', legend=False, ax=ax, showfliers=False)
ax.set_title('Customer Total Spend: One-Time vs Repeat Buyers (Excl. Outliers)', pad=12)
ax.set_xlabel('Loyalty Segment')
ax.set_ylabel('Total Monetary Spend (BRL)')
plt.show()